[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dilru1/microlensing-dtw/blob/main/notebooks/04_soft_dtw.ipynb)

# Step 4 — Soft-DTW: tuning how much to warp

Like Lab 2: soft-DTW's γ controls smoothness. On centred curves, sweep γ and see how t_E recovery changes.
Takes about 3 minutes.

In [ ]:
!pip -q install tslearn VBMicrolensing
!git clone -q https://github.com/Dilru1/microlensing-dtw.git
import sys; sys.path.insert(0, "microlensing-dtw")

import numpy as np
import matplotlib.pyplot as plt
from mldtw.simulate import make_dataset, CLASSES, T
from mldtw.bench import Bench
FIG_DIR = "microlensing-dtw/figures"

In [ ]:
# smaller set: soft-DTW is ~8x slower than DTW
Xtr, ytr, ptr = make_dataset(60, seed=1, classes=[2, 3])    # 120 events
Xte, yte, pte = make_dataset(25, seed=2, classes=[2, 3])    # 50 events
ttr = np.log10([p["tE"] for p in ptr])
tte = np.log10([p["tE"] for p in pte])

bench = Bench(Xtr, Xte).run(["Euclid, centred", "Centred + DTW"])
gammas = [0.01, 0.1, 1, 10]
for g in gammas:
    bench.add(f"soft-DTW g={g}", bench.soft(g))
rmse = bench.regress(ttr, tte)
rmse

In [ ]:
plt.semilogx(gammas, [rmse[f"soft-DTW g={g}"] for g in gammas], "o-", label="centred soft-DTW")
plt.axhline(rmse["Euclid, centred"], ls="--", c="k", label="Euclid, centred")
plt.axhline(rmse["Centred + DTW"], ls=":", c="r", label="Centred + DTW")
plt.xlabel("gamma"); plt.ylabel("tE RMSE [dex]"); plt.legend()
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/step4_soft_dtw.png", dpi=150)

**Read the result:** small γ behaves like DTW. Does larger γ, which spreads the alignment over many paths, move the error toward the shift-only baseline?

**Next directions (for the proposal)**
- A model with the right invariance built in: e.g. a CNN or attention encoder on centred, log-time curves.
- Use that encoder as the summary network for neural posterior estimation of (t₀, t_E, u₀, ρ).
- Add binary-lens (planetary) events and real OGLE/KMTNet light curves.